<a href="https://colab.research.google.com/github/thanmayeekotha27/flyrank-ml-internship/blob/main/notebooks/01_first_look_and_discovery.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 1 — Run it, then discover a real truth yourself

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thanmayeekotha27/flyrank-ml-internship/blob/main/notebooks/01_first_look_and_discovery.ipynb?flush_cache=true)

By the end of this notebook you will have:
1. **Run a real ML pipeline** on real (anonymized) search data and watched a learned model beat a hand-written rule.
2. **Rediscovered a real finding yourself** in ~10 lines of pandas.

No prior ML needed. Everything runs on the small anonymized dataset that ships with this repo — no credentials, no private data.

## 0. Setup (Colab or local)
On Colab this clones the repo and installs requirements. Locally it just moves to the repo root.

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

## 1. Run the whole pipeline

This runs `scripts/run_all.py`: prepare features → baseline rule → train 3 models → evaluate → PDF.
It takes ~1 minute on the 30,000-row sample.

In [ ]:
# Watch it work — each of the 5 steps prints live as it runs (~1 minute total).
!{sys.executable} scripts/run_all.py


### What just happened?
The pipeline ranked every page for "refresh review" two ways: a **hand-written rule baseline** and a **learned model**. Let's compare them on **Precision@50** — of the top 50 pages each says to fix first, how many are actually declining?

In [ ]:
import json
res = json.load(open("outputs/model_results.json"))

base = res["baseline"]["baseline_precision_at_50"]
rf   = res["models"]["random_forest"]["precision_at_50"]

print(f"Hand-written rule  Precision@50: {base:.3f}   (~{round(base*50)} of the top 50 right)")
print(f"Random forest      Precision@50: {rf:.3f}   (~{round(rf*50)} of the top 50 right)")
print(f"\nThe learned model roughly {rf/base:.1f}x the rule on this metric.")
print("Validation split used:", res["split_strategy"], "(pages from a client are never in both train and test)")

You just ran a real ML system on real search data and saw a learned ranking beat a fixed rule. Now open `outputs/model_report.md` and skim it — that Markdown report is the *shape* of what your own capstone should produce.

## 2. Discover a real truth yourself

The safest, most satisfying early wins are **things you find in the data** — un-leakable, and they *are* the core lesson. Run the three cells below. Each is ~10 lines of pandas and each overturns a common SEO belief.

Every number is **computed live from the shipped CSV** — nothing is hardcoded.

In [ ]:
import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(df.shape[0], "rows,", df.shape[1], "columns")
df.head(3)

### Discovery A — "High search volume means more traffic." Does it?

In [ ]:
corr = df["search_volume"].corr(df["impressions_90d"])
print(f"Correlation between search_volume and impressions_90d: {corr:.3f}")
print("Near zero -> keyword search volume barely predicts the traffic a page actually gets.")

### Discovery B — the CTR cliff by position
Click-through rate is not flat: it collapses as you move down the results.

In [ ]:
visible = df[df["impressions_90d"] >= 100]
ctr_by_pos = visible.groupby("position_tier")["ctr"].mean().sort_values(ascending=False)
print(ctr_by_pos.round(4).to_string())
ctr_by_pos.plot(kind="bar", title="Mean CTR by position tier (impressions >= 100)", ylabel="CTR");

### Discovery C — is longer content the lever?
Compare word count for **declining** vs **growing** pages.

In [ ]:
wc = df.groupby("trend_direction")["word_count"].median()
print(wc.round(0).to_string())
print("\n'down' vs 'up' pages have almost the same median word count -> length is not the lever.")

## 3. 🔧 Your turn

Pick **one** of these and write a few lines below. This is your Week-1 discovery — you'll reference it in your Week-1 research-question write-up (on the InternHQ board).

- Redo Discovery A but only for pages with `impressions_90d > 0` — does the correlation change?
- In Discovery B, which `content_type` has the worst CTR *at the same position tier*?
- Find another belief to test: does `content_age_days` relate to `trend_direction`? Does `avg_position` relate to `ctr`?

**Rules:** describe what you observe as *observed / directional* — never "I proved Google's algorithm." Keep client data out of anything you publish.

In [ ]:
# Your discovery here


### Part 1: Redoing Discovery A (Only pages with impressions_90d > 0)
Let's see if excluding pages with zero impressions reveals a stronger correlation between search volume and actual traffic.

In [7]:
import os, sys, subprocess
import pandas as pd

# Auto-setup check if we are in Colab but not inside the repository directory
REPO_DIR = "flyrank-ml-internship-starter"
if "google.colab" in sys.modules and not os.path.exists("data/raw/content_refresh_anonymized.csv"):
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/flyrank-bih/flyrank-ml-internship-starter", REPO_DIR], check=True)
    os.chdir(REPO_DIR)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
corr = df["search_volume"].corr(df["impressions_90d"])

active_pages = df[df["impressions_90d"] > 0]
corr_active = active_pages["search_volume"].corr(active_pages["impressions_90d"])
print(f"Correlation (all pages): {corr:.3f}")
print(f"Correlation (impressions > 0): {corr_active:.3f}")

Correlation (all pages): 0.001
Correlation (impressions > 0): 0.001


### Part 2: Analyzing CTR by Content Type at the same Position Tier
Let's group by both `position_tier` and `content_type` to find which content types underperform on click-through rate.

In [8]:
import os, sys, subprocess
import pandas as pd

# Auto-setup check if we are in Colab but not inside the repository directory
REPO_DIR = "flyrank-ml-internship-starter"
if "google.colab" in sys.modules and not os.path.exists("data/raw/content_refresh_anonymized.csv"):
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/flyrank-bih/flyrank-ml-internship-starter", REPO_DIR], check=True)
    os.chdir(REPO_DIR)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

visible_types = df[df["impressions_90d"] >= 100]
ctr_by_type = visible_types.groupby(["position_tier", "content_type"])["ctr"].mean().reset_index()
# Sort to see the lowest CTRs within each position tier
display(ctr_by_type.sort_values(by=["position_tier", "ctr"]))

,position_tier,content_type,ctr
0,deep,feedly article,0.044000
1,deep,keyword article,0.055547
2,page_1,comparison article,0.141239
4,page_1,keyword article,0.345805
3,page_1,feedly article,0.904796
5,page_3_5,comparison article,0.093968
7,page_3_5,keyword article,0.142711
6,page_3_5,feedly article,0.165610
8,striking,comparison article,0.147368
10,striking,keyword article,0.255881


### Part 3: Does Content Age relate to Trend Direction?
Let's test if older pages are more likely to be declining ('down') compared to growing pages ('up').

In [9]:
import os, sys, subprocess
import pandas as pd

# Auto-setup check if we are in Colab but not inside the repository directory
REPO_DIR = "flyrank-ml-internship-starter"
if "google.colab" in sys.modules and not os.path.exists("data/raw/content_refresh_anonymized.csv"):
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/flyrank-bih/flyrank-ml-internship-starter", REPO_DIR], check=True)
    os.chdir(REPO_DIR)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

age_by_trend = df.groupby("trend_direction")["content_age_days"].median()
print("Median content age (days) by trend direction:")
print(age_by_trend.round(1).to_string())

Median content age (days) by trend direction:
trend_direction
down      216.0
flat      231.0
new       279.0
stable    300.0
up        291.5


### Save your work
**Colab:** *File → Save a copy in GitHub* (writes to your own repo — that's your submission) and *File → Save a copy in Drive* (so the session doesn't evaporate).

Next: `02_your_first_readable_model.ipynb` — where the model becomes a rule you can read.